# VoiceOfBank — 05 Sentiment Analysis (RoBERTa)
**Notebook 5 of 7** — Run on Google Colab (GPU required)

Transformer-based sentiment analysis using `cardiffnlp/twitter-roberta-base-sentiment-latest`.

**Why this model instead of FinBERT:**
FinBERT was trained on financial news text (earnings reports, analyst notes).
App reviews are short, informal, and opinionated — much closer to Twitter text.
RoBERTa trained on Twitter sentiment correctly classifies app review language
where FinBERT fails (e.g. 'I love this app' scored as neutral by FinBERT,
correctly scored as positive by this model at 98.8% confidence).

**Input:** `data/processed/reviews_vader.csv` (from Google Drive)

**Output:** `data/processed/reviews_bert.csv` (saved to Google Drive)

**Expected runtime:** 8-12 minutes on Colab T4 GPU


## 1. Install Packages

In [ ]:
import subprocess
subprocess.run(
    ['pip', 'install', 'transformers', 'torch', 'datasets', 'accelerate', '--quiet'],
    check=True
)
print('Packages ready.')


## 2. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
import json
import warnings
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — go to Runtime -> Change runtime type -> T4 GPU')

BANKS      = ['Monzo','Starling','Barclays','HSBC','NatWest','Lloyds']
CHALLENGER = ['Monzo','Starling']
PALETTE    = dict(zip(BANKS, sns.color_palette('tab10', 6)))
BATCH_SIZE = 64
MAX_LEN    = 128
MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment-latest'

print(f'Model  : {MODEL_NAME}')
print(f'Batch  : {BATCH_SIZE}')


## 3. Mount Google Drive and Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path if your Drive folder is different
DRIVE_PATH = '/content/drive/MyDrive/VoiceOfBank/data/processed/'
Path(DRIVE_PATH).mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DRIVE_PATH + 'reviews_vader.csv', parse_dates=['date'])

print('Shape:', df.shape)
print()
print('Sentiment distribution (star-based ground truth):')
for cls in ['Positive','Neutral','Negative']:
    n   = (df['sentiment']==cls).sum()
    pct = n / len(df) * 100
    print(f'  {cls:<10}: {n:>6,}  ({pct:.1f}%)')


## 4. Load RoBERTa Model

In [ ]:
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model     = model.to(DEVICE)
model.eval()

# id2label from model config: {0: negative, 1: neutral, 2: positive}
# Capitalise to match our convention
ROBERTA_LABELS = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

print('Model loaded.')
print(f'Parameters : {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M')
print(f'id2label   : {model.config.id2label}')
print(f'Our mapping: {ROBERTA_LABELS}')
print()

# Verify on a known positive and known negative review
tests = [
    ("I love this app, it is so easy to use and works perfectly", 'Expected: Positive'),
    ("Terrible app, keeps crashing and customer service is useless", 'Expected: Negative'),
    ("It is okay, does what I need it to do", 'Expected: Neutral'),
]
print('Verification tests:')
for text, expected in tests:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=1).cpu().numpy()[0]
    pred = ROBERTA_LABELS[probs.argmax()]
    conf = probs.max()
    print(f'  {expected}')
    print(f'  Predicted : {pred} ({conf:.1%} confidence)')
    print()


## 5. Inference Pipeline

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length     = self.max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt',
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
        }


def run_inference(texts, model, tokenizer, label_map, batch_size=64, max_len=128):
    dataset    = ReviewDataset(texts, tokenizer, max_len)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_labels = []
    all_probs  = []

    with torch.no_grad():
        for batch_num, batch in enumerate(dataloader):
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            probs          = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            preds          = probs.argmax(axis=1)
            all_labels.extend([label_map[p] for p in preds])
            all_probs.extend(probs.tolist())

            if (batch_num + 1) % 50 == 0:
                done = min((batch_num + 1) * batch_size, len(texts))
                print(f'  Processed {done:,} / {len(texts):,} reviews')

    return all_labels, all_probs


print('Inference function defined.')


## 6. Run Inference on All Reviews

In [ ]:
print(f'Running RoBERTa inference on {len(df):,} reviews...')
print(f'Batch size: {BATCH_SIZE} | Max length: {MAX_LEN} tokens')
print()

bert_labels, bert_probs = run_inference(
    df['text'].tolist(), model, tokenizer,
    label_map  = ROBERTA_LABELS,
    batch_size = BATCH_SIZE,
    max_len    = MAX_LEN,
)

df['bert_label']    = bert_labels
df['bert_prob_neg'] = [p[0] for p in bert_probs]
df['bert_prob_neu'] = [p[1] for p in bert_probs]
df['bert_prob_pos'] = [p[2] for p in bert_probs]

print()
print('RoBERTa label distribution:')
for cls in ['Positive','Neutral','Negative']:
    n   = (df['bert_label']==cls).sum()
    pct = n / len(df) * 100
    print(f'  {cls:<10}: {n:>6,}  ({pct:.1f}%)')
print()
print('Star rating distribution (ground truth):')
for cls in ['Positive','Neutral','Negative']:
    n   = (df['sentiment']==cls).sum()
    pct = n / len(df) * 100
    print(f'  {cls:<10}: {n:>6,}  ({pct:.1f}%)')


## 7. RoBERTa vs VADER vs Star Rating

In [ ]:
print('Performance comparison (vs star rating ground truth):')
print()
for name, preds in [('VADER', df['vader_label']), ('RoBERTa', df['bert_label'])]:
    acc = accuracy_score(df['sentiment'], preds)
    f1  = f1_score(df['sentiment'], preds, average='macro', zero_division=0)
    print(f'{name}:')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  Macro F1 : {f1:.4f}')
    print()

print('RoBERTa classification report:')
print(classification_report(
    df['sentiment'], df['bert_label'],
    labels=['Positive','Neutral','Negative'],
    zero_division=0
))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
labels = ['Positive','Neutral','Negative']

for i, (name, preds) in enumerate([('VADER', df['vader_label']),
                                     ('RoBERTa', df['bert_label'])]):
    cm = confusion_matrix(df['sentiment'], preds, labels=labels)
    cm_df = pd.DataFrame(cm,
        index   = [f'True {l[:3]}' for l in labels],
        columns = [f'Pred {l[:3]}' for l in labels]
    )
    acc = accuracy_score(df['sentiment'], preds)
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues',
                linewidths=0.5, ax=axes[i])
    axes[i].set_title(f'{name} — Accuracy {acc:.1%}', fontweight='bold')

plt.suptitle('VADER vs RoBERTa — Confusion Matrices', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Sentiment by Bank

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mean_pos = df.groupby('bank')['bert_prob_pos'].mean().sort_values(ascending=False)
colors   = ['#22c55e' if b in CHALLENGER else '#4f8ef7' for b in mean_pos.index]
axes[0].bar(mean_pos.index, mean_pos.values, color=colors, edgecolor='white')
axes[0].set_title('Mean RoBERTa Positive Probability by Bank', fontweight='bold')
axes[0].set_ylabel('Mean P(Positive)')
axes[0].set_ylim(0, 1)
for j, (bank, val) in enumerate(mean_pos.items()):
    axes[0].text(j, val+0.01, f'{val:.3f}', ha='center', fontsize=9)

neg_rate = df.groupby('bank').apply(
    lambda x: (x['bert_label']=='Negative').mean()*100
).sort_values(ascending=False)
colors2 = ['#22c55e' if b in CHALLENGER else '#4f8ef7' for b in neg_rate.index]
axes[1].bar(neg_rate.index, neg_rate.values, color=colors2, edgecolor='white')
axes[1].set_title('RoBERTa Negative Review Rate by Bank (%)', fontweight='bold')
axes[1].set_ylabel('% Reviews Classified Negative')
for j, (bank, val) in enumerate(neg_rate.items()):
    axes[1].text(j, val+0.3, f'{val:.1f}%', ha='center', fontsize=9)

sns.despine()
plt.suptitle('RoBERTa Sentiment — Challenger (green) vs Traditional (blue)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Sentiment summary per bank:')
summary = df.groupby('bank').agg(
    mean_pos_prob = ('bert_prob_pos', 'mean'),
    pct_positive  = ('bert_label', lambda x: (x=='Positive').mean()*100),
    pct_negative  = ('bert_label', lambda x: (x=='Negative').mean()*100),
).round(2).sort_values('mean_pos_prob', ascending=False)
summary['type'] = ['Challenger' if b in CHALLENGER else 'Traditional'
                   for b in summary.index]
print(summary.to_string())


## 9. Sentiment Drift Over Time

In [ ]:
df['year_month_dt'] = pd.to_datetime(df['year_month'])
monthly = df.groupby(['year_month_dt','bank'])['bert_prob_pos'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
for bank in BANKS:
    sub = monthly[monthly['bank']==bank].sort_values('year_month_dt')
    lw  = 2.5 if bank in CHALLENGER else 1.5
    ls  = '-'  if bank in CHALLENGER else '--'
    ax.plot(sub['year_month_dt'], sub['bert_prob_pos'],
            label=bank, color=PALETTE[bank], linewidth=lw,
            linestyle=ls, marker='o', markersize=3)

ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('Month')
ax.set_ylabel('Mean P(Positive) — RoBERTa')
ax.set_title('Monthly Sentiment Drift by Bank (RoBERTa)', fontweight='bold')
ax.legend(ncol=3, fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
sns.despine()
plt.tight_layout()
plt.show()

print('Look for dips corresponding to the review volume spikes in notebook 02.')
print('Barclays and Lloyds should show a notable dip around Feb-Apr 2026.')


## 10. Save

In [ ]:
df.to_csv(DRIVE_PATH + 'reviews_bert.csv', index=False)
print('Saved: reviews_bert.csv')
print(f'  Shape  : {df.shape}')
print()
new_cols = [c for c in df.columns if c.startswith('bert_')]
print('New columns added:')
for col in new_cols:
    print(f'  {col}')
print()
print('Download reviews_bert.csv from Drive and save to:')
print('  VoiceOfBank/data/processed/reviews_bert.csv')
print()
print('Next: 06_topic_modelling.ipynb  (run on Colab)')
